1. Setup patch dan model yoloV8

In [1]:
import cv2
from pathlib import Path
from ultralytics import YOLO

# Setup Path Root dan Model
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_PATH = ROOT / "runs" / "wastify_yolov8n" / "weights" / "best.pt"

if not MODEL_PATH.exists():
    MODEL_PATH = ROOT / "runs" / "detect" / "wastify_yolov8n" / "weights" / "best.pt"

print(f"[INFO] Memuat Model Terbaik dari: {MODEL_PATH}")
model = YOLO(str(MODEL_PATH))

# Daftar Kelas Sampah Wastify
print("[INFO] Kelas yang terdeteksi:", model.names)

[INFO] Memuat Model Terbaik dari: d:\project semester 5\TUGAS AKHIR\WASTIFY_PCV\runs\wastify_yolov8n\weights\best.pt
[INFO] Kelas yang terdeteksi: {0: 'logam', 1: 'paper', 2: 'plastic'}


2. Running demo webcam

In [3]:
# Inisialisasi Kamera Laptop (0 = Kamera Bawaan)
cap = cv2.VideoCapture(0)

# Atur Resolusi Layar Kamera (Opsional: 1280x720)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print("[INFO] Membuka Jendela Kamera Wastify...")
print("[INFO] Arahkan sampah Logam, Plastik, atau Kertas ke kamera.")
print("[INFO] Tekan tombol 'q' pada keyboard untuk menghentikan demo.")

# Warna Bounding Box untuk Masing-masing Kelas (BGR Format)
COLOR_MAP = {
    'logam': (0, 255, 255),    # Kuning untuk Logam
    'paper': (255, 191, 0),    # Biru Muda untuk Kertas/Paper
    'plastic': (0, 255, 0)     # Hijau untuk Plastik
}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("[ERROR] Gagal membaca frame dari kamera.")
        break

    # Run Inferensi YOLOv8 pada Frame Webcam (Threshold Confidence 40%)
    results = model(frame, conf=0.40)

    for result in results:
        for box in result.boxes:
            cls_id = int(box.cls[0].item())
            cls_name = model.names[cls_id].lower()
            conf = float(box.conf[0].item())
            
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            # Tentukan warna bounding box sesuai jenis sampah
            box_color = COLOR_MAP.get(cls_name, (255, 255, 255))
            
            # Gambar Kotak (Bounding Box) Sampah
            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 3)
            
            # Label Nama Kelas & Confidence
            label = f"Sampah {cls_name.capitalize()} ({conf*100:.1f}%)"
            
            # Latar belakang label biar mudah dibaca
            (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
            cv2.rectangle(frame, (x1, y1 - 30), (x1 + w, y1), box_color, -1)
            cv2.putText(frame, label, (x1, y1 - 8), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

    # Tampilkan Header Info di Layar Webcam
    cv2.rectangle(frame, (0, 0), ( frame.shape[1], 40), (40, 40, 40), -1)
    cv2.putText(frame, "WASTIFY - Demo Klasifikasi Sampah (Logam | Kertas | Plastik)", 
                (20, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # Tampilkan Window OpenCV
    cv2.imshow("Wastify Real-time Classification Demo", frame)

    # Tekan 'q' untuk keluar
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("[INFO] Demo Klasifikasi Webcam Selesai.")

KeyboardInterrupt: 